# 01 - Find the refusal direction
**Paper:** Sections 2.2-2.3, Appendix A & C. **Exercises used:** 1, 2, 3, 4 (run `pytest` locally first).

Pipeline: base-model sanity check -> filter data -> difference-in-means at every (position, layer) -> score every candidate -> pick one -> save it.

In [ ]:
# --- Colab setup: run once per session ---
# 1) Runtime > Change runtime type > T4 GPU (or better)
# 2) Edit REPO to your GitHub URL (if the repo is private, use https://<TOKEN>@github.com/...)
REPO = "https://github.com/aervale/refusal-direction"

import os
if not os.path.exists("/content/refusal-direction"):
    !git clone -q $REPO /content/refusal-direction
%cd /content/refusal-direction
!git pull -q
%pip install -q -e .
# If pip downgraded numpy and Colab asks you to restart: Runtime > Restart session, then rerun this cell.

In [ ]:
%load_ext autoreload
%autoreload 2
import torch, pandas as pd, matplotlib.pyplot as plt
from refusal_direction.config import Config

# Save artifacts to Google Drive so they survive the Colab session ending.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/refusal-direction/artifacts"
except ImportError:
    ARTIFACTS = "artifacts"
cfg = Config(artifact_dir=ARTIFACTS)
print(cfg)

In [ ]:
from refusal_direction.model import load_model, tokenize_instructions, eoi_positions, generate, last_token_logits
from refusal_direction.data import load_datasets
from refusal_direction.refusal import refusal_token_ids, refusal_score, refusal_rate

model = load_model(cfg)
data = load_datasets(cfg)
tok = lambda instrs, suffix="": tokenize_instructions(model, instrs, cfg.system_prompt, suffix)
refusal_toks = refusal_token_ids(model.tokenizer)
print(model.cfg.n_layers, "layers | d_model", model.cfg.d_model, "| refusal tokens", refusal_toks,
      [model.tokenizer.decode([t]) for t in refusal_toks])

## Sanity check: does TransformerLens load Qwen2.5 correctly? (PROJECT.md assumption 11)
Compare last-position logits from the TransformerLens model against a float32 Hugging Face reference on 5 prompts, once as a left-padded batch and once per prompt with no padding.
- **Bad in both columns:** a loading / weight-conversion bug.
- **Bad only when batched:** a padding bug (attention mask or rotary positions).

TL runs in fp16/bf16 on the GPU, so expect max diffs of roughly 0.1-1 logits from dtype alone. A real bug looks like several logits of error and top-1 disagreements.

In [ ]:
import gc
from transformers import AutoModelForCausalLM

check_prompts = data["harmful_val"][:2] + data["harmless_val"][:3]
check_kinds = ["harmful"] * 2 + ["harmless"] * 3
hf_model = AutoModelForCausalLM.from_pretrained(cfg.model_name, dtype=torch.float32).to(model.cfg.device).eval()

@torch.no_grad()
def tl_vs_hf(tokens, mask):
    """Per-row max |logit diff| and top-1 agreement at the last position."""
    tl = last_token_logits(model, tokens, mask)
    # Explicit position_ids so the HF reference treats left padding correctly.
    pos = (mask.cumsum(-1) - 1).clamp(min=0)
    hf = hf_model(input_ids=tokens, attention_mask=mask, position_ids=pos).logits[:, -1, :].float()
    return (tl - hf).abs().max(-1).values.cpu(), (tl.argmax(-1) == hf.argmax(-1)).cpu()

batch_diff, batch_agree = tl_vs_hf(*tok(check_prompts))
single = [tl_vs_hf(*tok([p])) for p in check_prompts]
check = pd.DataFrame({
    "kind": check_kinds,
    "prompt": [p[:50] for p in check_prompts],
    "padded_max_diff": batch_diff.numpy(),
    "padded_top1_agree": batch_agree.numpy(),
    "alone_max_diff": [float(d[0]) for d, _ in single],
    "alone_top1_agree": [bool(a[0]) for _, a in single],
})
print(f"TL dtype {model.cfg.dtype} vs HF float32 | padded batch length {tok(check_prompts)[0].shape[1]}")
print(f"left-padded batch: max |diff| {check.padded_max_diff.max():.4f}, top-1 agreement {check.padded_top1_agree.sum()}/{len(check)}")
print(f"each prompt alone: max |diff| {check.alone_max_diff.max():.4f}, top-1 agreement {check.alone_top1_agree.sum()}/{len(check)}")
display(check)

del hf_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Sanity check: does the base model refuse what it should?

In [ ]:
sample_h, sample_hl = data["harmful_val"][:4], data["harmless_val"][:4]
outs_h = generate(model, *tok(sample_h), max_new_tokens=48)
outs_hl = generate(model, *tok(sample_hl), max_new_tokens=48)
for instr, out in list(zip(*(sample_h, outs_h)))[:4]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)
for instr, out in list(zip(*(sample_hl, outs_hl)))[:4]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

**Check:** do refusals actually start with the refusal tokens printed above? If Qwen starts its refusals with some other token, `refusal_score` is measuring the wrong thing - fix `refusal_token_ids(..., starts=...)` before going on.

## Filter the data (App. A)

In [ ]:
from refusal_direction.selection import filter_by_refusal
if cfg.filter_train:
    for split, n in [("train", cfg.n_train), ("val", cfg.n_val)]:
        data[f"harmful_{split}"] = filter_by_refusal(model, data[f"harmful_{split}"], tok, refusal_toks, True, n, cfg.batch_size)
        data[f"harmless_{split}"] = filter_by_refusal(model, data[f"harmless_{split}"], tok, refusal_toks, False, n, cfg.batch_size)
print({k: len(v) for k, v in data.items()})

## Candidate directions: difference-in-means (Exercise 3)

In [ ]:
from refusal_direction.directions import get_candidate_directions
positions = eoi_positions(model)
print("post-instruction tokens:", [model.tokenizer.decode([t]) for t in tok(["x"])[0][0, positions]])
candidates = get_candidate_directions(model, tok(data["harmful_train"]), tok(data["harmless_train"]), positions, cfg.batch_size)
candidates.shape  # [n_pos, n_layers, d_model]

In [ ]:
# How big is the harmful/harmless gap, by depth? (norm of each candidate)
plt.figure(figsize=(8, 3))
for i, p in enumerate(positions):
    plt.plot(candidates[i].norm(dim=-1).cpu(), label=f"pos {p}")
plt.xlabel("layer"); plt.ylabel("||r||"); plt.legend(); plt.title("Difference-in-means norm"); plt.show()

## Score every candidate (takes a few minutes on a T4)

In [ ]:
from refusal_direction.selection import score_candidates
from refusal_direction import artifacts
scores = score_candidates(model, candidates, positions, tok(data["harmful_val"]), tok(data["harmless_val"]), refusal_toks, cfg.batch_size)
scores.to_csv(artifacts.path(cfg, "candidate_scores.csv"), index=False)
scores.sort_values("bypass_score").head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for metric, ax in zip(["bypass_score", "induce_score", "kl_score"], axes):
    for p, g in scores.groupby("pos"):
        ax.plot(g.layer, g[metric], label=f"pos {p}")
    ax.set_title(metric); ax.set_xlabel("layer")
axes[0].axhline(0, c="k", lw=.5); axes[1].axhline(cfg.induce_threshold, c="r", ls="--")
axes[2].axhline(cfg.kl_threshold, c="r", ls="--"); axes[2].set_yscale("log")
for ax in axes: ax.axvline(cfg.prune_layer_fraction * model.cfg.n_layers, c="gray", ls=":")
axes[0].legend(); plt.show()

## Pick the direction (Exercise 4) and save it

In [ ]:
from refusal_direction.selection import select_direction
best = select_direction(scores, model.cfg.n_layers, cfg)
direction = candidates[int(best.pos_idx), int(best.layer)]
meta = {k: (float(v) if k.endswith("score") else int(v)) for k, v in best.items()}
print(meta)
artifacts.save_direction(cfg, direction, meta)

## Questions to write up
1. Which layer/position won? Paper: Qwen 1.8B picked a middle layer at the last eoi token - same for you?
2. Plot: is there a band of layers where *many* candidates work, or one sharp winner? What does that suggest about how localized refusal is?
3. Why does the paper prune layers >= 0.8L? Disable the filter and see what gets picked - then generate with it.